In [6]:
# !pip3 install firebase-admin

In [1]:
import firebase_admin
from firebase_admin import credentials, firestore

In [2]:
# Path to your downloaded Firebase private key JSON file
cred = credentials.Certificate("C:\\Users\\CurveSystem 52\\Desktop\\IMRAN_WORK\\credentials\\health-app-7a8b0-firebase-adminsdk-fbsvc-9d28c0ae3f.json")

# Initialize Firebase App (only once per script)
firebase_admin.initialize_app(cred)

# Initialize Firestore client
db = firestore.client()

# Get all top-level collections
collections = db.collections()

print("Collections in Firestore:")
for collection in collections:
    print(collection.id)

Collections in Firestore:
admins
chatHistory
doctors
emergencyContacts
insurance
medicalReports
medicines
personalContacts
personalDetails
pharmacies
userPreferences
users


In [52]:
def get_user_medical_history_and_medicine_summary(uid: str):
    # ---------------------------
    # 1. Get medical history
    # ---------------------------
    user_ref = db.collection("users").document(uid)
    doc = user_ref.get()

    if not doc.exists:
        return {"error": "User not found"}

    data = doc.to_dict()
    name=data.get("displayName")
    
    summaries = []
    if "answers" in data:
        for item in data["answers"]:
            sum_ans = item.get("summarizedAnswer")
            if sum_ans:
                summaries.append(sum_ans)

    # ---------------------------
    # 3. Get chat history
    # ---------------------------
    ref = db.collection("chatHistory")
    query = ref.where("userId", "==", uid)

    messages = []

    for doc in query.stream():
        data = doc.to_dict()
        # print(data)
        sender = "user" if data.get("isUser") else "bot"

        messages.append({
            "message": data.get("message"),
            "sender": sender
        })


    # ---------------------------
    # 4. Build final response
    # ---------------------------
    return {
        "user_id": uid,
        'user_name':name,
        "medical_history": summaries,
        "chat_history":messages
    }


# Call function
result = get_user_medical_history_and_medicine_summary("pGSZNP7t8odnxmOIpRbDcuthbbM2")

print(result)

# print(result["user_id"])
# print("\n")
# print(result["medical_history"])
# print("\n")
# print(result["medicines"])

{'user_id': 'pGSZNP7t8odnxmOIpRbDcuthbbM2', 'user_name': 'Scott Grody', 'medical_history': ["Patient's date of birth is 07/06/1946.", 'Patient identifies as Male.', "Patient's ethnicity is Caucasian .", "Patient's address is 21377 Gosier Way.", 'Patient resides in Boca Raton.', 'Patient resides in Florida .', "Patient's zip code is 33428.", "Patient's phone number is 5617025533.", 'Patient\'s height is 5\'10".', "Patient's weight is 175 lbs.", 'Patient has had surgeries in the past.', 'Laser on left vocal cord ', 'Patient has been hospitalized in the past.', '2012 for vocal cord', 'Patient has high blood pressure.', 'Patient provided details about blood pressure.', 'Patient has diabetes.', 'Patient provided details about diabetes.', 'Patient does not have heart disease.', 'Patient does not have any allergies.', 'Patient currently smokes tobacco.', '1/2 pack a day', 'Patient consumes alcohol.', 'Socially ', 'Patient does not use recreational drugs.', 'Patient has not experienced recent 

In [53]:
# -----------------------
# Get all chatHistory documents
# -----------------------
def get_user_messages_clean(user_id: str):
    ref = db.collection("chatHistory")
    query = ref.where("userId", "==", user_id)

    messages = []

    for doc in query.stream():
        data = doc.to_dict()
        # print(data)
        sender = "user" if data.get("isUser") else "bot"

        messages.append({
            "message": data.get("message"),
            "sender": sender
        })

    return messages


chats = get_user_messages_clean("pGSZNP7t8odnxmOIpRbDcuthbbM2")
for chat in chats:
    print(chat)

{'message': 'Take care, Scott! If you have any health concerns in the future, feel free to reach out. Wishing you well!', 'sender': 'bot'}
{'message': 'i have headache right now', 'sender': 'user'}
{'message': 'Scott, When did this start, and what makes it better or worse?', 'sender': 'bot'}
{'message': 'Thank you for sharing that, Scott. Is the pain localized to a specific area of your back, or does it feel more generalized?', 'sender': 'bot'}
{'message': 'Hi Scott! How can I assist you with your health today?', 'sender': 'bot'}
{'message': 'How long has your runny nose been bothering you, Scott?', 'sender': 'bot'}
{'message': "I'm having back pain and did not do anything strenuous ", 'sender': 'user'}
{'message': 'Scott, When did this start, and what makes it better or worse?', 'sender': 'bot'}
{'message': "I'm having trouble right now. Please try again.\n\nPossible Causes\n- Several common causes may fit based on your description.\n\nWhat You Can Do Now\n- Rest, hydrate, and avoid k

In [46]:
def get_recent_chat_history(uid: str, limit: int = 10):
    """Get recent chat history - last N messages only"""
    try:
        ref = db.collection("chatHistory")
        query = ref.where("userId", "==", uid).order_by(
            "timestamp", direction=firestore.Query.DESCENDING
        ).limit(limit)
        
        messages = []
        for doc in query.stream():
            data = doc.to_dict()
            sender = "user" if data.get("isUser") else "bot"
            messages.append({
                "message": data.get("message", ""),
                "sender": sender,
                "timestamp": data.get("timestamp")
            })
        
        # Reverse to get chronological order
        messages.reverse()
        return messages
    except Exception as e:
        # Silently fail - chat history is optional
        return []


messages=get_recent_chat_history("pGSZNP7t8odnxmOIpRbDcuthbbM2")
print(messages)

[]


In [51]:
# ref = db.collection("chatHistory")
# query = ref.where("userId", "==", "pGSZNP7t8odnxmOIpRbDcuthbbM2")
# messages = []
# for doc in query.stream():
#     data = doc.to_dict()
#     sender = "user" if data.get("isUser") else "bot"
#     messages.append({
#         "message": data.get("message", ""),
#         "sender": sender,
#         "timestamp": data.get("timestamp")
#     })

# # Reverse to get chronological order
# messages.reverse()
# print(messages)